# MatchGPT-paradigm baseline (LLM paradigm)

**Purpose**: validate the 5-50 gap against the LLM paradigm. Does zero-shot or few-shot ICL LLM prompting bypass the warm-start gap, or hit similar constraints?

**Reference**: Peeters & Bizer 2024, [github.com/wbsg-uni-mannheim/MatchGPT/tree/main/LLMForEM](https://github.com/wbsg-uni-mannheim/MatchGPT/tree/main/LLMForEM)

**No code clone needed** — this notebook only calls LLM APIs (Claude, OpenAI, or Together/LLaMA). The MatchGPT *paradigm* (zero-shot LLM prompting + optional related-demos retrieval) is implemented directly here.

## Provider options + cost summary

| Provider | Model | Cost (in/out per 1M tokens) | Zero-shot 4 targets | Few-shot K=10 4 targets |
|---|---|---|---|---|
| Anthropic | claude-sonnet-4-6 | $3 / $15 | ~$5 | ~$60 |
| OpenAI | gpt-4o-mini | $0.15 / $0.60 | ~$0.50 | ~$3 |
| OpenAI | gpt-4o | $2.50 / $10 | ~$4 | ~$40 |
| Together.ai | meta-llama/Llama-3.1-70B-Instruct-Turbo | $0.88 / $0.88 | ~$1.65 | ~$17 |
| Groq | llama-3.1-70b-versatile | free tier available | ~free | ~free with rate-limiting |

**Recommended**: start with Claude Sonnet zero-shot on Pair #1 only (~$0.20) as a sanity check, then scale up.


## 1. Bootstrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = os.environ.get('REPO_ROOT') or '/content/drive/MyDrive/cd-er-paradigm-choice'
os.environ['REPO_ROOT'] = BASE
os.chdir(BASE)
print('cwd:', os.getcwd())

!git config user.email "author041@gmail.com"
!git config user.name "author97"
!git pull --ff-only || echo '[bootstrap] git pull failed; continuing'


## 2. Install dependencies + set API key

The cell installs `anthropic`, `openai`, and (if you plan to use the "related demonstrations" strategy) `sentence-transformers`. The provider client is selected via the `PROVIDER` toggle in cell 4.

**Set your API key** via Colab Secrets (recommended) or environment variable. Examples:

```python
# Option A — Colab Secrets (gear icon → Secrets → add ANTHROPIC_API_KEY)
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

# Option B — inline (not recommended — visible in notebook)
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
```


In [ ]:
!pip install -q anthropic openai sentence-transformers

# Set API key from Colab Secrets — change provider name to match what you'll use
from google.colab import userdata
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('ANTHROPIC_API_KEY: set')
except Exception:
    print('ANTHROPIC_API_KEY: not set (skip if using OpenAI/Together)')

try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('OPENAI_API_KEY: set')
except Exception:
    print('OPENAI_API_KEY: not set (skip if using Anthropic/Together)')

try:
    os.environ['TOGETHER_API_KEY'] = userdata.get('TOGETHER_API_KEY')
    print('TOGETHER_API_KEY: set')
except Exception:
    print('TOGETHER_API_KEY: not set (skip if using Anthropic/OpenAI)')


## 3. Configure provider, model, targets, and prompting strategy

`DEMO_STRATEGY` controls demonstration selection for few-shot mode:
- `"random"`: K random stratified demos (half positive, half negative) selected once and reused
- `"related"`: K nearest neighbours to the query pair by sentence-embedding similarity (their headline contribution; requires `sentence-transformers`)

`USE_THEIR_DATA = True` clones their `LLMForEM` repo and uses their downsampled test splits for direct comparison to published F1 numbers. Default `False` uses our standard Ditto-format splits for consistency with §V.E.


In [ ]:
# ────────────────────────────────────────────────────────────────────────
# CONFIG — edit these as needed
# ────────────────────────────────────────────────────────────────────────
# PROVIDER = "anthropic"          # "anthropic" | "openai" | "together"
# MODEL = "claude-sonnet-4-6"      # see provider docs for current names
PROVIDER = "together"
MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

# What to run
RUN_ZERO_SHOT = True
RUN_FEW_SHOT_K = [10]            # K∈{6,10} matches their tables; [] = skip few-shot
DEMO_STRATEGY = "random"         # "random" | "related"

# Targets — comment out pairs you've already done
TARGETS = [
    # "Textual/Abt-Buy",                     # Pair #1
    # "wdc/watches",                         # Pair #2
    # "Structured/DBLP-ACM",                 # Pair #3
    # "Structured/Amazon-Google",            # Pair #5
    "Dirty/DBLP-ACM"
]

# True = use MatchGPT's downsampled splits (for direct paper replication)
# False = use our Ditto-format splits (for §V.E consistency)
USE_THEIR_DATA = False

SKIP_IF_DONE = True


## 4. Provider-agnostic LLM wrapper

The same prompt + model interface works across Anthropic, OpenAI, and Together via OpenAI-compatible APIs. The `call_llm(system, user)` function is the unified entry point used by the run loop.


In [ ]:
if PROVIDER == "anthropic":
    import anthropic
    client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
    def call_llm(system, user, max_tokens=10):
        msg = client.messages.create(
            model=MODEL,
            max_tokens=max_tokens,
            system=system,
            messages=[{"role": "user", "content": user}],
        )
        return msg.content[0].text.strip()

elif PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
    def call_llm(system, user, max_tokens=10):
        resp = client.chat.completions.create(
            model=MODEL,
            max_tokens=max_tokens,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}],
        )
        return resp.choices[0].message.content.strip()

elif PROVIDER == "together":
    from openai import OpenAI
    client = OpenAI(api_key=os.environ.get('TOGETHER_API_KEY'),
                    base_url="https://api.together.xyz/v1")
    def call_llm(system, user, max_tokens=10):
        resp = client.chat.completions.create(
            model=MODEL,
            max_tokens=max_tokens,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}],
        )
        return resp.choices[0].message.content.strip()

else:
    raise ValueError(f"Unknown PROVIDER: {PROVIDER}")

print(f"[ready] PROVIDER={PROVIDER}  MODEL={MODEL}")


## 5. Prompts (zero-shot and few-shot ICL)

The templates match the MatchGPT paper's wording. For few-shot mode, demonstrations are listed before the query pair with `Yes`/`No` answers.


In [ ]:
SYSTEM_PROMPT = (
    "You are a data integration expert specialising in product entity "
    "matching. For each pair of product descriptions, decide whether "
    "they refer to the same real-world product. Reply with exactly "
    "'Yes' or 'No' — no other text."
)

ZERO_SHOT_TEMPLATE = (
    "Do the two product descriptions refer to the same product?\n\n"
    "Product 1: {left}\n\n"
    "Product 2: {right}\n\n"
    "Answer (Yes/No):"
)

FEW_SHOT_INTRO = (
    "Below are {K} example pairs labeled by an expert. Study them, then "
    "answer the final pair using the same Yes/No format."
)
FEW_SHOT_EXAMPLE = (
    "Example {i}:\nProduct 1: {left}\nProduct 2: {right}\nAnswer: {answer}\n"
)
FEW_SHOT_QUERY = (
    "\nNow the question:\nProduct 1: {left}\nProduct 2: {right}\nAnswer (Yes/No):"
)

def build_few_shot_prompt(demos, query_pair, k):
    parts = [FEW_SHOT_INTRO.format(K=k)]
    for i, d in enumerate(demos, 1):
        parts.append(FEW_SHOT_EXAMPLE.format(
            i=i, left=d['left'], right=d['right'],
            answer="Yes" if d['label'] == 1 else "No"))
    parts.append(FEW_SHOT_QUERY.format(left=query_pair['left'], right=query_pair['right']))
    return "\n".join(parts)


## 6. Data loading + demo selection

In [ ]:
import random
from pathlib import Path

def load_ditto_split(path):
    pairs = []
    with open(path) as f:
        for line in f:
            parts = line.rstrip().split("\t")
            if len(parts) >= 3:
                pairs.append({"left": parts[0], "right": parts[1], "label": int(parts[2])})
    return pairs

def get_test_and_demo_pools(target_dataset, use_their_data=False):
    """Return (test_pairs, demo_pool). demo_pool drawn from training split."""
    if use_their_data:
        # Their downsampled splits at MatchGPT/LLMForEM/data/<dataset>/
        their_name_map = {
            "Textual/Abt-Buy": "abtbuy",
            "Structured/Walmart-Amazon": "walmartamazon",
            "Structured/Amazon-Google": "amazongoogle",
            "wdc/computers": "wdcproducts",
        }
        their_name = their_name_map.get(target_dataset)
        if their_name is None:
            raise ValueError(f"No their-data mapping for {target_dataset}")
        # If MatchGPT repo isn't cloned, clone it first (~30 MB)
        matchgpt_dir = Path('/content/MatchGPT')
        if not matchgpt_dir.exists():
            import subprocess
            subprocess.run(['git', 'clone', '--depth', '1',
                            'https://github.com/wbsg-uni-mannheim/MatchGPT',
                            str(matchgpt_dir)], check=True)
        base = matchgpt_dir / "LLMForEM" / "data" / their_name
        test = load_ditto_split(base / "test.txt")
        train = load_ditto_split(base / "train.txt")
    else:
        # Our standard Ditto splits
        from config.config import PATHS
        test_path = PATHS.ditto_repo / "data" / "er_magellan" / target_dataset / "test.txt"
        train_path = PATHS.ditto_repo / "data" / "er_magellan" / target_dataset / "train.txt"
        if not test_path.exists():
            # WDC isn't under er_magellan
            cat = target_dataset.split("/")[-1]
            for variant in ("", "_large", "_xlarge", "_medium", "_small"):
                test_path = PATHS.ditto_repo / "data" / "wdc" / f"{cat}{variant}" / "test.txt"
                if test_path.exists():
                    train_path = PATHS.ditto_repo / "data" / "wdc" / f"{cat}{variant}" / "train.txt"
                    break
        test = load_ditto_split(test_path)
        train = load_ditto_split(train_path)
    return test, train

def select_random_demos(demo_pool, k, seed=42):
    """K random demos, stratified to keep half pos and half neg."""
    rng = random.Random(seed)
    pos = [d for d in demo_pool if d['label'] == 1]
    neg = [d for d in demo_pool if d['label'] == 0]
    n_pos = k // 2
    n_neg = k - n_pos
    return (rng.sample(pos, min(n_pos, len(pos)))
            + rng.sample(neg, min(n_neg, len(neg))))


## 7. Related-demo retrieval (required only for DEMO_STRATEGY='related')

Pre-embeds the full demo pool once per target using sentence-transformers, then retrieves the K most similar demos per query by cosine similarity. This implements MatchGPT's headline "related demonstrations" strategy (Table 5 in their paper).


In [ ]:
if DEMO_STRATEGY == "related":
    from sentence_transformers import SentenceTransformer
    import numpy as np

    EMBEDDER = SentenceTransformer('all-MiniLM-L6-v2')
    print(f"[loaded] sentence-transformers/all-MiniLM-L6-v2")

    def _pair_text(p):
        return f"{p['left']} [SEP] {p['right']}"

    def embed_pool(demo_pool):
        texts = [_pair_text(p) for p in demo_pool]
        emb = EMBEDDER.encode(texts, batch_size=64, show_progress_bar=False,
                              convert_to_numpy=True, normalize_embeddings=True)
        return emb, demo_pool

    def select_related_demos(query_pair, pool_emb, pool_pairs, k):
        query_emb = EMBEDDER.encode([_pair_text(query_pair)],
                                     normalize_embeddings=True, convert_to_numpy=True)[0]
        sims = pool_emb @ query_emb
        order = np.argsort(-sims)
        pos_picks, neg_picks = [], []
        n_pos = k // 2
        n_neg = k - n_pos
        for idx in order:
            p = pool_pairs[idx]
            if p['label'] == 1 and len(pos_picks) < n_pos:
                pos_picks.append(p)
            elif p['label'] == 0 and len(neg_picks) < n_neg:
                neg_picks.append(p)
            if len(pos_picks) >= n_pos and len(neg_picks) >= n_neg:
                break
        return pos_picks + neg_picks
else:
    embed_pool = None
    select_related_demos = None
    print(f"[skip] DEMO_STRATEGY={DEMO_STRATEGY}, sentence-transformers not needed")


## 8. Main run loop

For each (target, setting) combination:
1. Load test pairs + demo pool
2. Pre-embed pool if DEMO_STRATEGY=='related'
3. For each test pair: build prompt, call LLM, parse Yes/No
4. Compute F1, precision, recall against ground truth
5. Save metrics.json + predictions.jsonl

Skip-if-done logic: a successful (test_f1 != None, returncode == 0) run is not re-executed.


In [ ]:
import json, time
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score
from config.config import PATHS

def run_setting(target_dataset, setting_name, K, output_dir):
    test_pairs, demo_pool = get_test_and_demo_pools(target_dataset, use_their_data=USE_THEIR_DATA)
    pos_rate = sum(p['label'] for p in test_pairs) / len(test_pairs)
    print(f"\n[{target_dataset} / {setting_name}]  {len(test_pairs)} test pairs  "
          f"(positives: {sum(p['label'] for p in test_pairs)}, pos_rate={pos_rate:.3f}, demos: {K})")

    pool_emb = pool_pairs_ref = None
    if K > 0 and DEMO_STRATEGY == "related":
        print(f"  [embed] computing pool embeddings for {len(demo_pool)} demo pairs ...")
        pool_emb, pool_pairs_ref = embed_pool(demo_pool)

    static_demos = None
    if K > 0 and DEMO_STRATEGY == "random":
        static_demos = select_random_demos(demo_pool, K)

    preds_path = output_dir / 'predictions.jsonl'
    start = time.time()
    y_true, y_pred = [], []

    with open(preds_path, 'w') as out_f:
        for i, p in enumerate(test_pairs):
            if K == 0:
                user_msg = ZERO_SHOT_TEMPLATE.format(**p)
            elif DEMO_STRATEGY == "related":
                demos = select_related_demos(p, pool_emb, pool_pairs_ref, K)
                user_msg = build_few_shot_prompt(demos, p, K)
            else:
                user_msg = build_few_shot_prompt(static_demos, p, K)

            try:
                text = call_llm(SYSTEM_PROMPT, user_msg, max_tokens=10).lower()
            except Exception as e:
                print(f"  [{i}] ERROR: {e}")
                text = ""
            pred = 1 if text.startswith("yes") else 0
            y_true.append(p['label']); y_pred.append(pred)
            out_f.write(json.dumps({"idx": i, "label": p['label'], "pred": pred,
                                     "raw": text}) + "\n")
            if (i+1) % 100 == 0:
                rate = (i+1) / (time.time() - start)
                eta = (len(test_pairs) - i - 1) / rate / 60
                print(f"  [{i+1}/{len(test_pairs)}]  {rate:.1f} pairs/sec  ETA: {eta:.1f} min")

    elapsed = time.time() - start
    f1 = f1_score(y_true, y_pred)
    p_score = precision_score(y_true, y_pred, zero_division=0)
    r_score = recall_score(y_true, y_pred, zero_division=0)
    print(f"  done in {elapsed/60:.1f} min — F1={f1:.4f}  P={p_score:.4f}  R={r_score:.4f}")

    (output_dir / 'metrics.json').write_text(json.dumps({
        "method": f"matchgpt-{PROVIDER}",
        "model": MODEL,
        "dataset": target_dataset,
        "setting": setting_name,
        "k_demos": K,
        "demo_strategy": DEMO_STRATEGY if K > 0 else "n/a",
        "n_test_pairs": len(test_pairs),
        "positive_rate": pos_rate,
        "use_their_data": USE_THEIR_DATA,
        "elapsed_sec": elapsed,
        "test_f1": float(f1),
        "test_precision": float(p_score),
        "test_recall": float(r_score),
        "returncode": 0,
    }, indent=2))
    return f1, elapsed

# Build settings list
SETTINGS = []
if RUN_ZERO_SHOT:
    SETTINGS.append(('zero_shot', 0))
for k in RUN_FEW_SHOT_K:
    SETTINGS.append((f'few_shot_k{k}_{DEMO_STRATEGY}', k))

results = {}
for target in TARGETS:
    for setting_name, K in SETTINGS:
        safe_target = target.replace("/", "__")
        data_suffix = "_their_data" if USE_THEIR_DATA else ""
        out_dir = PATHS.results_runs / f"matchgpt-{PROVIDER}{data_suffix}" / safe_target / setting_name
        out_dir.mkdir(parents=True, exist_ok=True)

        if SKIP_IF_DONE and (out_dir / 'metrics.json').exists():
            existing = json.loads((out_dir / 'metrics.json').read_text())
            if existing.get('test_f1') is not None and existing.get('returncode', 0) == 0:
                print(f"[skip] {target} / {setting_name} — already done F1={existing['test_f1']:.4f}")
                continue

        f1, elapsed = run_setting(target, setting_name, K, out_dir)
        results[(target, setting_name)] = (f1, elapsed)

print("\n=== Summary ===")
for (target, setting), (f1, elapsed) in results.items():
    print(f"  {target:35s}  {setting:20s}  F1={f1:.4f}  ({elapsed/60:.1f} min)")


## 9. Sanity check against published numbers (MatchGPT paper)

Their published F1 ranges from Tables 2-5:

| Setting | Best F1 | Dataset | Model |
|---|---|---|---|
| Zero-shot | 95.78 | Abt-Buy | GPT-4 |
| Zero-shot | 89.82 | DBLP-Scholar | GPT-4 |
| Zero-shot | 98.41 | DBLP-ACM | GPT-4 |
| Few-shot K=10 (related) | 94.87 | Abt-Buy | GPT-4o |
| Few-shot K=10 (related) | 91.74 | WDC Products | GPT-4o |

If your Claude/LLaMA result is within ±5pp of these on the matching dataset, the prompt format is correct.


In [ ]:
# Quick validation against published F1 ranges
PUBLISHED = {
    ('Textual/Abt-Buy', 'zero_shot'): (0.85, 0.96),       # GPT-4 zero-shot range
    ('Structured/DBLP-ACM', 'zero_shot'): (0.88, 0.99),    # GPT-4
    ('Textual/Abt-Buy', 'few_shot_k10'): (0.85, 0.96),     # GPT-4o related demos
}

print('\n=== Validation against published F1 ===\n')
data_suffix = "_their_data" if USE_THEIR_DATA else ""
RUNS_DIR = PATHS.results_runs / f"matchgpt-{PROVIDER}{data_suffix}"
for target in TARGETS:
    safe = target.replace("/", "__")
    for setting_name, K in SETTINGS:
        m_path = RUNS_DIR / safe / setting_name / 'metrics.json'
        if not m_path.exists():
            continue
        m = json.loads(m_path.read_text())
        f1 = m.get('test_f1')
        # Strip the strategy suffix for lookup
        lookup_setting = setting_name.split('_random')[0].split('_related')[0]
        exp_range = PUBLISHED.get((target, lookup_setting))
        if f1 is None:
            verdict = '❌ no F1 — check predictions.jsonl'
        elif exp_range is None:
            verdict = '⚪ no published reference for this combination'
        elif exp_range[0] - 0.05 <= f1 <= exp_range[1] + 0.05:
            verdict = f'✓ within ±5pp of published [{exp_range[0]}, {exp_range[1]}]'
        else:
            verdict = f'⚠ outside published range [{exp_range[0]}, {exp_range[1]}]'
        print(f'  {target:35s} {setting_name:20s} F1={f1:.4f}  {verdict}')


## 10. Push results to git

In [ ]:
!git add results/runs/matchgpt-*/
!git status --short


In [ ]:
!git commit -m "MatchGPT baseline: zero-shot + few-shot K=10 across 4 targets" || echo 'nothing to commit'
!git push


## 11. Cost estimate review (run AFTER your runs complete)

The total per-target API cost depends on test-set size and few-shot K. This cell aggregates actual elapsed_sec and estimated cost from completed runs.


In [ ]:
# Aggregate runs and report observed cost / runtime
import json
from pathlib import Path

data_suffix = "_their_data" if USE_THEIR_DATA else ""
RUNS_DIR = PATHS.results_runs / f"matchgpt-{PROVIDER}{data_suffix}"

# Approximate per-1M-token cost (rough, edit per provider/model)
COST_PER_M_TOKENS = {
    'anthropic': {'in': 3.0, 'out': 15.0},
    'openai': {'in': 2.5, 'out': 10.0},
    'together': {'in': 0.88, 'out': 0.88},
}

total_cost = 0.0
print(f'\n=== Cost + runtime summary ({PROVIDER}/{MODEL}) ===\n')
print(f'  {"target":35s}  {"setting":20s}  {"F1":>6}  {"min":>6}  {"est. $":>8}')
print('-' * 90)
for target in TARGETS:
    safe = target.replace('/', '__')
    for setting_name, K in SETTINGS:
        m_path = RUNS_DIR / safe / setting_name / 'metrics.json'
        if not m_path.exists():
            continue
        m = json.loads(m_path.read_text())
        f1 = m.get('test_f1', 0) or 0
        elapsed_min = (m.get('elapsed_sec') or 0) / 60
        # Estimate tokens per pair: zero-shot ~250 in/5 out; K=10 ~2800 in/5 out
        tok_in_per_pair = 250 if K == 0 else (250 + K * 250)
        tok_out_per_pair = 5
        n = m.get('n_test_pairs', 0) or 0
        c = COST_PER_M_TOKENS.get(PROVIDER, {'in': 0, 'out': 0})
        est_cost = (tok_in_per_pair * n * c['in'] + tok_out_per_pair * n * c['out']) / 1_000_000
        total_cost += est_cost
        print(f'  {target:35s}  {setting_name:20s}  {f1:.4f}  {elapsed_min:6.1f}  ${est_cost:7.2f}')

print('-' * 90)
print(f'  TOTAL estimated cost across reported runs:  ${total_cost:.2f}')


## After Pair #1, extending to other pairs and few-shot mode

Once Pair #1 zero-shot validates (~$0.20, ~20 min wall-clock for Claude on AB):

1. Uncomment the other targets in cell 4's `TARGETS` list
2. Optionally enable few-shot: `RUN_FEW_SHOT_K = [10]` (costs ~10x zero-shot due to demo tokens)
3. Optionally switch to `DEMO_STRATEGY = "related"` (their headline contribution, ~3pp F1 lift)
4. Re-run cells 4 → 8 (skip-if-done protects completed runs)